# sLLM: Llama 계열 모델을 RunPod GPU에서 추론하기

**sLLM(Small Language Model)** 은 제한된 GPU·CPU·메모리에서 직접 실행하거나 특정 업무에 맞게 조정할 수 있는 비교적 작은 언어 모델이다.

sLLM, SLM, LLM을 가르는 공식 파라미터 기준은 없으나 대략 다음과 같이 판단한다.

- **LLM (대형 언어 모델):** 보통 70B ~ 수천B+ 이상 (대형)
- **sLLM (소형 거대 언어 모델):** 보통 7B ~ 14B 내외 (중소형)
- **SLM (소형 언어 모델):** 보통 수억 ~ 3B/7B 이하 (초소형)

관리형 LLM은 공급자가 모델과 GPU를 운영하지만, 로컬 sLLM/SLM은 사용자가 모델 weight, 실행 장치, cache와 생성 설정을 직접 관리한다.

---

### 주요 특성

sLLM은 상대적으로 적은 메모리와 연산량으로 실행할 수 있어 비용과 응답 지연을 줄이고, 온프레미스나 엣지 환경에서 데이터를 직접 관리하는 데 유리하다. 특정 업무 데이터로 조정하면 범용 지식보다 한정된 도메인 작업에 집중할 수 있다.

- **경량화**: 수십억~수백억 규모의 파라미터로 구성되어, LLM 대비 메모리·연산 요구량이 대폭 감소한다.
- **비용 효율성**: 학습·추론 비용이 낮고, 전력 소모가 적어 온프레미스나 엣지 환경에서 활용하기 적합하다.
- **실시간성**: 파라미터 수가 적어 추론 속도가 빠르며, 모바일·노트북·임베디스 환경에서도 운영 가능하다.
- **도메인 특화**: 특정 분야 데이터로 미세조정(fine-tuning)하면 LLM과 유사한 성능을 달성할 수 있다.

    - 온프레미스(On-Premise): 외부 클라우드가 아닌 회사 자체 전산실에 물리 서버를 직접 두고 데이터를 안전하게 관리·운영하는 방식이다.
    - 엣지(Edge): 중앙 서버를 거치지 않고 데이터가 발생하는 사용자 말단 기기(스마트폰, IoT, PC 등) 자체에서 즉시 연산·처리하는 환경이다.

---

### 활용 기술


- **양자화(Quantization)**: weight의 수치 정밀도를 낮춰 모델 크기와 VRAM 사용량을 줄이는 방법이다.
- **LoRA(Low-Rank Adaptation)**: 전체 weight 대신 작은 저랭크 행렬을 학습해 파인튜닝 비용을 줄이는 방법이며 다음 단원에서 실습한다.
- **지식 증류(Knowledge Distillation)**: 큰 교사 모델의 출력이나 확률 분포를 작은 학생 모델이 학습하게 하는 방법이다.

---

### 한계

모델이 작아지면 복잡한 다단계 추론과 폭넓은 지식 처리에서 큰 모델보다 성능이 낮을 수 있다. 한국어처럼 상대적으로 학습 데이터가 적은 언어는 데이터 구성에 따라 품질 차이가 커질 수 있으며, 모델 크기가 작다고 환각이 사라지는 것은 아니므로 중요한 답변은 별도로 검증해야 한다.


## Hugging Face에서 모델을 받기 전에 알아둘 것

**Hugging Face Model Hub**는 모델의 코드만 보관하는 곳이 아니라 tokenizer 설정, model config, weight 파일과 사용 설명을 repository 단위로 제공하는 저장소이다. `meta-llama/Llama-3.1-8B-Instruct`처럼 `제작자/모델명`으로 구성된 문자열을 **model ID**라고 한다. 이어지는 `from_pretrained()`는 이 ID를 사용해 필요한 파일을 내려받고 local cache에 보관한다.

모델 페이지의 **model card**에서는 다음 항목을 먼저 확인한다.

- 모델 크기와 목적: Llama 3.1 Instruct는 8B 대화 모델이고, Bllossom은 Llama 3.2 3B를 한국어·영어로 강화한 모델이다.
- 라이선스와 사용 조건: 두 모델은 각각 Llama 3.1, Llama 3.2 라이선스를 따르므로 배포·상업 이용 전에 조건을 확인한다.
- 입력 형식: 대화 모델은 model repository에 포함된 chat template으로 `user`와 `assistant` 역할을 token으로 바꾼다.
- 저장 공간과 VRAM: 8B는 약 80억 개 parameter라는 뜻이며, 16-bit weight만 계산해도 약 16GB이다. 실제 실행에는 cache와 중간 tensor 공간이 추가로 필요하다.

[`meta-llama/Llama-3.1-8B-Instruct`](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)는 **gated model**이다. gated model은 파일을 받기 전에 Hugging Face 계정으로 model page의 이용 조건에 동의하고 접근 권한을 받아야 한다. token은 승인된 계정임을 증명하지만 접근 승인을 대신하지 않는다.

[`Bllossom/llama-3.2-Korean-Bllossom-3B`](https://huggingface.co/Bllossom/llama-3.2-Korean-Bllossom-3B)도 실행 전에 model card와 현재 접근 상태를 확인한다. 이 실습에서는 Llama 3.1 8B를 먼저 실행하고 메모리에서 내린 뒤 Bllossom 3B를 실행한다.


## Hugging Face Access Token 발급하기

`HF_TOKEN`은 Hugging Face 계정의 접근 권한을 Python Notebook에 전달하는 **User Access Token**이다. Llama 3.1처럼 gated model을 내려받으려면 모델 접근 승인과 token이 모두 필요하며 학생마다 자신의 계정으로 준비한다.

1. [Hugging Face](https://huggingface.co/)에 회원가입하고 로그인한다.
2. [Llama 3.1 8B Instruct 모델 페이지](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)에서 이용 조건을 확인하고 접근을 요청한다. 자동 승인이 아니면 승인까지 시간이 걸릴 수 있으므로 수업 전에 진행한다.
3. 접근이 허용되면 `프로필 이미지 → Settings → Access Tokens`로 이동한다.
4. `Create new token` 또는 `New token`을 누르고 이름을 `runpod-class`처럼 구분 가능한 값으로 정한다.
5. 모델 다운로드만 수행하므로 `read` 권한을 선택한다. 더 좁은 권한을 설정할 수 있다면 Llama repository 읽기만 허용하는 fine-grained token을 사용할 수 있다.
6. 생성된 `hf_...` 값을 한 번 복사해 안전한 곳에 보관한다. token은 비밀번호와 같은 자격증명이므로 다른 학생과 공유하지 않는다.

공식 절차는 [Hugging Face Gated Models](https://huggingface.co/docs/hub/models-gated)와 [User Access Tokens](https://huggingface.co/docs/hub/security-tokens) 문서에서 확인할 수 있다.


## RunPod에 HF_TOKEN과 OPENAI_API_KEY 등록하기

RunPod에는 key 값을 평문으로 직접 저장하지 않고 **Secret을 환경변수에 연결**한다. `HF_TOKEN`은 Llama model 다운로드에 필요하고, `OPENAI_API_KEY`는 마지막 관리형 LLM 비교 셀을 실행할 때만 필요하다. OpenAI 비교를 실행하지 않으면 `OPENAI_API_KEY`는 생략할 수 있으며 API 사용료는 RunPod 비용과 별도로 발생한다.

### Secret 만들기

1. RunPod Console에서 `Secrets → Create Secret`으로 이동한다.
2. Secret 이름 `SKN_33_HF_TOKEN`에 앞에서 발급한 Hugging Face token을 저장한다.
3. OpenAI 비교도 실행한다면 Secret 이름 `SKN_33_OPENAI_API_TOKEN`에 [OpenAI API key](https://platform.openai.com/api-keys)를 저장한다.
4. Secret을 생성하면 실제 값은 화면에서 다시 보이지 않는다. 값이 잘못되었으면 Secret의 `Manage → Edit Secret Value`에서 교체한다.

### 실행 중인 Pod에 연결하기

1. `Pods`에서 대상 Pod의 점 3개 메뉴를 열고 `Edit Pod`를 선택한다.
2. `Environment Variables`를 펼쳐 다음 두 항목을 추가한다. Secret 선택 아이콘이 보이지 않으면 아래 참조식을 값에 직접 입력한다.

```text
HF_TOKEN={{ RUNPOD_SECRET_SKN_33_HF_TOKEN }}
OPENAI_API_KEY={{ RUNPOD_SECRET_SKN_33_OPENAI_API_TOKEN }}
HF_HOME=/workspace/cache/huggingface
```

`HF_HOME`은 비밀값이 아니라 Hugging Face model cache를 `/workspace`에 보존하기 위한 경로 설정이다.

3. `Save`를 누르면 Pod가 재시작된다. `/tmp/pycharm_project_*`를 포함해 `/workspace` 또는 Network Volume 밖의 데이터가 지워질 수 있으므로 필요한 파일을 먼저 `/workspace`에 저장한다.
4. Pod가 다시 실행되면 현재 Host와 SSH port를 확인하고 PyCharm SSH를 재연결한다. 이미 열려 있던 Notebook kernel도 재시작하거나 원격 kernel을 다시 선택해야 새 환경변수를 읽는다.

새 Pod를 만들 때는 배포 화면의 `Edit Template → Environment Variables`에 같은 매핑을 넣는다. 여러 수업에서 반복 사용한다면 `My Templates`의 비공개 template에 Secret 참조만 저장한다. RunPod Secret을 사용하면 프로젝트에 `.env`를 업로드할 필요가 없다.

실제 token이나 key를 `print()`, `echo`, Notebook 출력 또는 Git에 남기지 않는다. 자세한 내용은 [RunPod Secrets](https://docs.runpod.io/pods/templates/secrets), [Environment Variables](https://docs.runpod.io/pods/templates/environment-variables), [Pod 편집 시 초기화 범위](https://docs.runpod.io/pods/manage-pods)에서 확인할 수 있다.


## 패키지 설치

이 셀은 네트워크를 사용해 Llama 계열 모델과 OpenAI 비교에 필요한 패키지를 설치한다. Llama 3.1의 공식 model card는 대화형 추론에 `transformers>=4.43.0`을 안내한다. `torch`는 앞 단원에서 준비한 RunPod PyTorch 환경을 그대로 사용한다.

- `transformers`: tokenizer, model loading과 text generation을 제공한다.
- `accelerate`: `device_map='auto'`에 따라 사용 가능한 GPU에 모델을 배치하도록 돕는다.
- `huggingface-hub`: model repository 다운로드와 gated model 인증을 담당한다.
- `openai`: 마지막에 관리형 LLM을 호출할 때 사용한다.


In [1]:
%pip install -U "transformers==4.56.2" "accelerate>=1,<2" "typing_extensions==4.16.0" "openai==2.15.0" hf_transfer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 65.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.4 MB/s  0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.032m 0/11 [typing_extensions]
  Attempting uninstall: huggingface-hubm━━━━━━━━━━━━━━━━━━━━━  5/11 [pydantic-core]
    Found existing installation: huggingface_hub 1.28.0━━━━━━━  5/11 [pydantic-core]
    Uninstalling huggingface_hub-1.28.0:m━━━━━━━━━━━━━━━━━━━━━  5/11 [pydantic-core]
      Successfully uninstalled huggingface_hub-1.28.0━━━━━━━━━  5/11 [pydantic-core]
  Attempting uninstall: transformersm╺━━━━━━━━━━━━━━  7/11 

In [2]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


## 환경변수와 모델 ID 설정

RunPod가 Python process에 주입한 `HF_TOKEN`을 읽고 두 Hugging Face model ID를 변수로 지정한다. `HF_HOME`은 Hugging Face library가 자동으로 읽으므로 코드에서 다시 지정하지 않는다.


In [3]:
import os
import torch

HF_TOKEN = os.getenv('HF_TOKEN')

LLAMA31_MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
BLOSSOM_MODEL_ID = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

for name in ("RUNPOD_POD_ID", "HF_TOKEN", "OPENAI_API_KEY"):
    print(name, "설정됨" if os.getenv(name) else "없음")


RUNPOD_POD_ID 설정됨
HF_TOKEN 설정됨
OPENAI_API_KEY 설정됨


## tokenizer와 model을 불러오는 함수

`AutoTokenizer`는 문자열과 token ID를 변환하며 model repository의 chat template과 special token 정보도 읽는다. `AutoModelForCausalLM`은 앞 token을 바탕으로 다음 token을 생성하는 causal language model weight를 불러온다.

`torch_dtype`은 weight를 GPU에 올릴 정밀도이다. Llama 3.1 8B에는 `torch.float16`, Bllossom 3B에는 `torch.bfloat16`을 전달한다. `device_map='auto'`는 Accelerate가 사용 가능한 GPU에 모델을 배치하게 하고, `token`에는 환경변수에서 읽은 Hugging Face token을 전달한다.


In [4]:

def load_local_model(model_id: str, torch_dtype: torch.dtype):
    # token은 Llama 3.1 gated repository의 접근 권한을 증명한다.
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=HF_TOKEN,
    )

    # device_map='auto'는 사용 가능한 RunPod GPU에 weight를 배치한다.
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN,
        torch_dtype=torch_dtype,
        device_map='auto',
    )
    return tokenizer, model

## chat template 기반 생성 함수

대화 모델은 일반 문자열보다 `user`·`assistant` 역할과 special token이 포함된 입력 형식을 기대한다.

`apply_chat_template()`은 메시지 목록을 `input_ids`와 `attention_mask`가 담긴 batch 형태로 바꾼다. `attention_mask`는 실제 입력 token과 padding 위치를 구분한다.

`add_generation_prompt=True`는 assistant 응답 시작 위치를 추가한다. `return_dict=True`를 사용하면 `input_ids`와 `attention_mask`를 함께 받아 attention mask가 누락될 때 나타나는 경고를 막을 수 있다. Llama 계열 repository에 따라 문장 전체 종료 token인 `<|end_of_text|>`와 대화 turn 종료 token인 `<|eot_id|>`를 함께 사용할 수 있으므로, 존재하는 token ID를 종료 조건으로 전달한다. `generation_options`에는 모델별 sampling 설정을 넘긴다.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def generate_by_sllm(
    prompt: str,
    tokenizer,
    model,
    max_new_tokens: int = 500,
    **generation_options,
) -> str:
    messages = [{'role': 'user', 'content': prompt}]

    # return_dict=True는 input_ids와 attention_mask를 함께 담은 BatchEncoding을 반환한다.
    # return_tensors='pt'는 두 값을 PyTorch tensor로 만들어 model 입력에 연결한다.
    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
    ).to(model.device)

    # 두 모델이 사용하는 문장 종료와 대화 turn 종료 token을 생성 종료 조건으로 지정한다.
    terminator_ids = [
        tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
        tokenizer.convert_tokens_to_ids('<|eot_id|>'),
    ]

    # **model_inputs는 attention mask까지, **generation_options는 모델별 생성 설정을 전달한다.
    # max_new_tokens는 새 답변 길이를 제한하고 eos_token_id는 turn 종료 시 생성을 멈춘다.
    # pad_token_id는 별도 padding token이 없는 Llama 모델에서 eos token을 대신 사용한다.
    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminator_ids,
            pad_token_id=tokenizer.eos_token_id,
            **generation_options,
        )

    # batch 0의 입력 길이 뒤만 잘라 새 assistant token을 복원한다.
    input_length = model_inputs['input_ids'].shape[-1]
    generated_ids = output_ids[0, input_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

## Llama 3.1 8B 모델을 한 번만 적재하기

이 셀부터 Hugging Face model 다운로드와 GPU 메모리 사용이 시작된다. 같은 tokenizer와 model 객체를 영어·한국어 질문에 재사용해 weight를 반복해서 적재하지 않는다.


In [6]:
local_tokenizer, local_model = load_local_model(
    LLAMA31_MODEL_ID,
    torch.float16,
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

### 영어 질문 실행

영어 질문을 Llama 3.1에 전달한다. `do_sample=False`는 확률 sampling 대신 가장 가능성이 높은 token을 이어 붙이는 greedy decoding을 사용한다.


In [7]:
ENGLISH_PROMPT = 'Explain the difference between high and low tides.'

english_answer = generate_by_sllm(
    ENGLISH_PROMPT,
    local_tokenizer,
    local_model,
    do_sample=False,
)
print(english_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


High and low tides are two distinct phases of the ocean's water level, caused by the gravitational pull of the moon and the sun on the Earth's oceans. Here's a detailed explanation of the difference between high and low tides:

**High Tide:**

High tide occurs when the gravitational pull of the moon and the sun on the Earth's oceans is at its strongest, causing the water level to rise. This happens when the moon is directly overhead or directly below a particular location on Earth. The combined gravitational force of the moon and the sun pulls the water towards them, creating a bulge in the ocean. This bulge is what we experience as high tide.

**Low Tide:**

Low tide, on the other hand, occurs when the gravitational pull of the moon and the sun on the Earth's oceans is at its weakest, causing the water level to drop. This happens when the moon is at a 90-degree angle to a particular location on Earth, or when the sun's gravitational pull is opposing the moon's. As a result, the water 

### 같은 모델에 한국어 질문 실행

가중치를 다시 적재하지 않고 한국어 질문을 같은 Llama 3.1 tokenizer와 model에 전달한다. 영어와 한국어 생성 결과를 관찰하되, 두 문장만으로 모델의 전체 언어 성능을 단정하지 않는다.


In [8]:
KOREAN_PROMPT = '하늘 빛은 왜 파란거야? 한글로 대답해줘~'

korean_answer = generate_by_sllm(
    KOREAN_PROMPT,
    local_tokenizer,
    local_model,
    do_sample=False,
)
print(korean_answer)

## Llama 3.2 기반 Bllossom 모델 실행하기

두 번째 모델인 Bllossom 3B를 실행한다. 두 모델을 동시에 GPU에 올리면 OOM이 발생할 수 있으므로 먼저 Llama 3.1 객체 참조를 해제하고 CUDA cache를 정리한 뒤 Bllossom을 적재한다.

Bllossom에는 연필 계산 질문과 model card 예시의 sampling 설정을 사용한다. `temperature=0.6`은 token 확률 분포의 무작위성을, `top_p=0.9`는 누적 확률 90% 안의 후보만 사용하는 범위를 결정한다. 종료 token과 attention mask는 공통 함수에서 전달해 경고와 불필요한 장문 생성을 막는다.


In [9]:
import gc

# Llama 3.1 답변 문자열은 남기고 GPU weight 참조만 해제한다.
local_tokenizer = None
local_model = None
gc.collect()
torch.cuda.empty_cache()

korean_tokenizer, korean_model = load_local_model(
    BLOSSOM_MODEL_ID,
    # bfloat16은 Bllossom 예시에서 사용하는 weight 정밀도이다.
    torch.bfloat16,
)

# 한국어 산술 지시로 단계별 계산 능력을 확인한다.
BLOSSOM_PROMPT = (
    '철수가 20개의 연필을 가지고 있었는데 영희가 절반을 가져가고 '
    '민수가 남은 5개를 가져갔으면 철수에게 남은 연필의 갯수는 몇개인가요?'
)
# max_new_tokens는 새 답변의 상한이고, do_sample=True는 확률 sampling을 켠다.
# temperature와 top_p는 model card 예시의 sampling 범위를 재현한다.
korean_specialized_answer = generate_by_sllm(
    BLOSSOM_PROMPT,
    korean_tokenizer,
    korean_model,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print(korean_specialized_answer)

## 관리형 LLM과 실행 방식 비교

관리형 API는 모델 weight와 GPU를 직접 운영하지 않고 공급자의 endpoint에 요청한다. 이 비교를 진행하려면 RunPod 환경변수에 `OPENAI_API_KEY`가 있어야 한다. 다음 셀은 `OpenAI` client와 Chat Completions 호출 함수를 준비하며, 실제 요청은 그다음 셀에서 실행한다.


In [12]:
from openai import OpenAI

# OpenAI()는 RunPod 환경변수의 OPENAI_API_KEY를 자동으로 사용한다.
client = OpenAI()

def generate_by_llm(prompt: str, model: str = 'gpt-5.6-luna') -> str:
    # model은 호출할 관리형 모델 ID이고 messages는 user 역할과 질문을 전달한다.
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        # max_completion_tokens=5000
    )
    return response.choices[0].message.content

### 관리형 모델 요청 실행하기

아래 셀을 실행하면 실제 네트워크 요청과 토큰 비용이 발생한다. 로컬 Llama 실습을 마친 뒤 관리형 모델과 실행 책임의 차이를 비교할 때만 실행한다.


In [13]:
model_answer = generate_by_llm("만약에 달이 없으면 무슨 일이 일어나?")
print(model_answer)

달이 **갑자기 사라진다**고 가정하면, 지구가 즉시 산산조각 나지는 않지만 장기적으로 큰 변화가 생깁니다.

1. **조수간만의 차가 크게 약해짐**  
   현재 바닷물의 조수는 주로 달의 중력 때문에 생깁니다. 달이 없어지면 태양이 만드는 조수만 남아, 조수간만의 차가 지금보다 훨씬 작아집니다.  
   → 갯벌·조간대 생태계, 해안 생물의 번식과 먹이사슬이 크게 영향을 받습니다.

2. **해안 생태계가 붕괴할 수 있음**  
   게, 조개, 갯지렁이, 철새 등 조수에 적응한 생물들이 타격을 받습니다. 바닷물의 혼합과 영양분 순환도 달라져 해양 생태계가 변할 수 있습니다.

3. **지구 자전 변화 양상이 달라짐**  
   달의 조석 마찰은 지구의 자전을 아주 조금씩 느리게 하고 있습니다. 달이 없으면 이 작용이 크게 줄어들어, 장기적으로 하루 길이가 늘어나는 속도가 훨씬 느려집니다. 다만 달이 사라지는 순간 하루가 갑자기 짧아지지는 않습니다.

4. **지구 자전축이 불안정해질 가능성**  
   달은 지구 자전축의 기울기를 안정시키는 역할을 합니다. 달이 없으면 수만~수백만 년에 걸쳐 자전축 기울기가 크게 변동할 수 있습니다.  
   → 계절의 세기와 기후가 훨씬 불안정해지고, 극심한 빙하기나 온난기가 반복될 가능성이 커집니다.

5. **밤이 더 어두워짐**  
   보름달빛이 사라져 야행성 동물, 곤충, 산호 등 달의 주기를 이용하는 생물들의 행동과 번식 주기가 바뀝니다. 인간의 밤하늘도 훨씬 어두워집니다.

6. **일식과 월식이 사라짐**  
   달이 없으므로 일식과 월식은 더 이상 일어나지 않습니다.

7. **달이 지구에 떨어지는 일은 없음**  
   달이 사라진다고 해서 지구의 중력이 갑자기 크게 약해지거나 지구가 태양으로 떨어지지는 않습니다. 달의 질량은 지구보다 매우 작고, 지구의 태양 공전도 거의 그대로입니다.

요약하면, **단기적으로는 조수와 밤의 밝기가 가장 크게 변하고, 장기적으로는 해양 생태계와 기후 안정성이 크게

## 정리

Hugging Face 접근 승인과 RunPod Secret의 `HF_TOKEN`을 사용해 Llama 3.1 8B를 실행한 뒤 GPU 메모리를 정리하고, Llama 3.2 기반 Bllossom 3B의 한국어 지시 수행을 확인했다. `HF_HOME`을 `/workspace` 아래에 두면 같은 Network Volume에서 model cache를 재사용할 수 있다. 마지막 관리형 LLM 셀은 외부 요청과 비용이 발생하므로 비교가 필요할 때만 실행한다.

다음 단원에서는 이미 학습된 base model 전체를 다시 학습하는 방식과 일부 작은 parameter만 학습하는 PEFT를 구분하고, LoRA가 weight 변화량을 낮은 rank의 두 행렬로 표현하는 이유를 확인한다.


In [11]:
import sys
import torch
